# RappiPlus — Análisis de negocio y comportamiento de usuarios

Proyecto de análisis de datos desarrollado con **Python, SQL, estadística y Power BI**.

Esta versión fue preparada para portafolio: se eliminaron credenciales y elementos propios de la entrega académica, se documentaron las decisiones de calidad de datos y se conservaron las consultas SQL relevantes como evidencia metodológica.

> Algunas fuentes SQL pertenecían al entorno original del bootcamp y no se distribuyen públicamente. Por ello, las secciones dependientes de esas tablas se documentan sin afirmar reproducibilidad donde los datos fuente ya no están disponibles.


---

## 🔹 Paso 1: Carga y validación de datos

### 1.1 Carga y exploración inicial

**Objetivo:** comprender la estructura de los datasets antes del análisis.

Se utilizan tres fuentes principales:
- `rappiplus_orders_raw.csv`: pedidos y ventas.
- `rappiplus_catalog.csv`: catálogo y costos de productos.
- `rappiplus_marketing_spend.csv`: inversión de marketing.

A partir de estas fuentes se realiza una exploración inicial de estructura, tipos de datos y calidad de la información.


In [1]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats import pointbiserialr

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv") 
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")
checkout = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")

In [3]:
# explorar datasets

orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
catalog.head()


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [5]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**Objetivo:** detectar y corregir problemas que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los tres datasets para:
- validar y convertir fechas;
- revisar variables numéricas;
- verificar consistencia de montos;
- identificar y eliminar duplicados;
- revisar variables categóricas y valores nulos.


**LIMPIEZA DE DATOS**

**Revisión de variables numericas y Verificación de montos**

In [6]:
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [7]:
ceros_negativos = orders[orders['cantidad'] <= 0].shape[0]
ceros_negativos

4

In [8]:
orders[orders['cantidad'] <= 0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


**Se detectaron 4 valores negativos, pero debido a que podrían tratarse de devoluciones, no serán eliminados del dataset para un posterior análisis**

In [9]:
orders['monto_esperado'] = (orders['precio_unitario'] * orders['cantidad']) - orders['monto_descuento']
orders['diferencia'] = orders['monto_total'] - orders['monto_esperado']

In [10]:
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,665.38,-0.01
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,171.86,0.00
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,195.98,0.01
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,242.87,0.00
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,336.28,0.00


In [11]:
inconsistencias = orders[orders['diferencia'] !=0]
inconsistencias

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,665.38,-0.01
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,195.98,0.01
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.19,327.18,0.01
7,order_7,user_1083,2025-05-01,Mexico,desktop,organic,Tablet-Standard-64GB,Electronica,2.0,373.68,0.0,747.35,747.36,-0.01
18,order_18,user_7316,2025-01-23,Mexico,mobile,social,Phone-Pro-128GB,Electronica,2.0,180.64,5.0,356.27,356.28,-0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25077,order_15310,user_3224,2025-01-02,Mexico,mobile,organic,Blender-XL-Red,Hogar,2.0,73.20,0.0,146.41,146.40,0.01
25079,order_2470,user_2056,2025-02-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,471.43,0.0,942.85,942.86,-0.01
25091,order_21779,user_3295,2025-05-11,Argentina,desktop,organic,Laptop-Gaming-16GB,Electronica,2.0,338.67,0.0,677.33,677.34,-0.01
25092,order_24367,user_434,2025-03-22,Colombia,desktop,organic,Laptop-Gaming-16GB,Electronica,2.0,216.57,15.0,418.13,418.14,-0.01


In [12]:
inconsistencias_positivas = orders[orders['diferencia'] > 0.02]
inconsistencias_positivas

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62,-212.62,20.0
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50,-48.50,10.0
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65,-502.65,10.0


In [13]:
inconsistencias_negativas = orders[orders['diferencia'] < -0.02]
inconsistencias_negativas

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia


**Se realizó la revisión de la consistencia de los valores para determinar si el monto total coincide con la cantidad real de la venta de los productos (cantidad * precio_unitario - descuento)**

 -En la mayoría de las inconsistencias si se detectaron diferencias entre el monto total y el monto esperado, pero la diferencia solamente va del -0.01 a 0.01 centavos, ya que esa diferencia es despreciable que no se realizará ningun ajuste.
 
 -Se detectaron 3 valores mayores a 0.01 centavos, pero provienen de lo que posiblemnete sean devoluciones, por lo que no se realizará alguna modificacion al dataset 


**Verificación de valores duplicados**

In [14]:
orders.duplicated().sum()

100

In [15]:
orders=orders.drop_duplicates()

In [16]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25000 non-null  object 
 1   id_usuario          25000 non-null  object 
 2   fecha_hora_pedido   25000 non-null  object 
 3   pais                24700 non-null  object 
 4   dispositivo         24980 non-null  object 
 5   fuente_referencia   24970 non-null  object 
 6   nombre_producto     24970 non-null  object 
 7   categoria_producto  24920 non-null  object 
 8   cantidad            24950 non-null  float64
 9   precio_unitario     24950 non-null  float64
 10  monto_descuento     24950 non-null  float64
 11  monto_total         25000 non-null  float64
 12  monto_esperado      24950 non-null  float64
 13  diferencia          24950 non-null  float64
dtypes: float64(6), object(8)
memory usage: 2.9+ MB


**Se eliminaron 100 filas duplicadas**

**Se corrige el formato de fecha ya que aparece como objet**

In [17]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])

**Se revisan variables categoricas**

In [18]:
orders['pais'].value_counts()

Colombia     7481
Mexico       7478
Argentina    7259
mexico        863
colombia      823
argentina     796
Name: pais, dtype: int64

In [19]:
orders["pais"] = orders["pais"].str.lower()
orders['pais'].value_counts(dropna=False)

mexico       8341
colombia     8304
argentina    8055
NaN           300
Name: pais, dtype: int64

**Se homologaron los nombres de los países**

    -Se detectaron 300 valores nulos.
    -Realizando una revisión de cada usuario, se determino que el pais es unico para cada usuario, por lo cual se realizara el método de mapeo para rellenar los valores nulos

In [20]:

mapeo_pais = orders.dropna(subset=['pais']).groupby('id_usuario')['pais'].first()
mapeo_pais


id_usuario
user_0          mexico
user_1          mexico
user_10      argentina
user_100     argentina
user_1001     colombia
               ...    
user_995     argentina
user_996     argentina
user_997     argentina
user_998      colombia
user_999     argentina
Name: pais, Length: 7624, dtype: object

In [21]:
orders['pais'] = orders['pais'].fillna(orders['id_usuario'].map(mapeo_pais))

In [22]:
orders['pais'].value_counts(dropna=False)

mexico       8440
colombia     8386
argentina    8155
NaN            19
Name: pais, dtype: int64

**Realizando el método de mapeo y rellenando los valores, se obtuvo una disminucion de 300 a 19 valores nulos en la columna de pais**

In [23]:
orders['dispositivo'].value_counts(dropna=False)

desktop    12711
mobile     12269
NaN           20
Name: dispositivo, dtype: int64

**Para el tipo de dispositivo tambien se detectaron valores nulos, se verificara si es posible utilizar el método anterior para disminuir los valores nulos**

In [24]:

dispositivos_por_usuario = orders.groupby('id_usuario')['dispositivo'].nunique().sort_values(ascending=False)
dispositivos_por_usuario


id_usuario
user_0       1
user_5791    1
user_5802    1
user_5801    1
user_5800    1
            ..
user_3376    1
user_3375    1
user_999     1
user_579     0
user_458     0
Name: dispositivo, Length: 7642, dtype: int64

In [25]:
dispositivos_por_usuario[dispositivos_por_usuario > 1]

Series([], Name: dispositivo, dtype: int64)

**Se realizó la comprobacion que los usuarios no utilizaron mas de 1 dispositivo para las compras realizadas, por lo cual es posible utilizar un mapeo para rellenar los valores nulos**

In [26]:
mapeo_dispositivo = orders.dropna(subset=['dispositivo']).groupby('id_usuario')['dispositivo'].first()
mapeo_dispositivo

id_usuario
user_0        mobile
user_1        mobile
user_10      desktop
user_100     desktop
user_1001    desktop
              ...   
user_995     desktop
user_996      mobile
user_997      mobile
user_998      mobile
user_999     desktop
Name: dispositivo, Length: 7640, dtype: object

In [27]:
orders['dispositivo'] = orders['dispositivo'].fillna(orders['id_usuario'].map(mapeo_dispositivo))

In [28]:
orders['dispositivo'].value_counts(dropna=False)

desktop    12721
mobile     12277
NaN            2
Name: dispositivo, dtype: int64

**Realizando el método de mapeo y rellenando los valores, se obtuvo una disminucion de 20 a 2 valores nulos en la columna de dispositivo**

In [29]:
orders['fuente_referencia'].value_counts(dropna=False)

social         8402
organic        8288
paid_search    8280
NaN              30
Name: fuente_referencia, dtype: int64

**Se encuentran 30 valores nulos, se analizara si es posible imputar, si no es posible se mantendran los valores nulos hasta algun proceso posterior**

In [30]:
orders['nombre_producto'].value_counts(dropna=False)

Blender-XL-Red          4179
Jacket-Winter-M         4176
Vacuum-Pro-Black        4176
Sneakers-Urban-42       4148
Laptop-Gaming-16GB      2782
Tablet-Standard-64GB    2769
Phone-Pro-128GB         2740
NaN                       30
Name: nombre_producto, dtype: int64

**Se encuentran 30 valores nulos, se analizara si es posible imputar, si no es posible se mantendran los valores nulos hasta algun proceso posterior**

In [31]:
orders['categoria_producto'].value_counts(dropna=False)

Hogar          8346
Moda           8295
Electronica    8279
NaN              80
Name: categoria_producto, dtype: int64

**Se encuentran 80 valores nulos, se analizara si es posible imputar, si no es posible se mantendran los valores nulos hasta algun proceso posterior**

In [32]:
categorias_por_producto = orders.groupby('nombre_producto')['categoria_producto'].nunique().sort_values(ascending=False)
categorias_por_producto

nombre_producto
Blender-XL-Red          1
Jacket-Winter-M         1
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Sneakers-Urban-42       1
Tablet-Standard-64GB    1
Vacuum-Pro-Black        1
Name: categoria_producto, dtype: int64

**Debido a que pudimos comprobar que cada producto pertenece a una sola categoría, es posible imputar los valores con el método de mapeo usado en los pasos anteriores**

In [33]:
mapeo_categoria = orders.dropna(subset=['categoria_producto']).groupby('nombre_producto')['categoria_producto'].first()
mapeo_categoria

nombre_producto
Blender-XL-Red                Hogar
Jacket-Winter-M                Moda
Laptop-Gaming-16GB      Electronica
Phone-Pro-128GB         Electronica
Sneakers-Urban-42              Moda
Tablet-Standard-64GB    Electronica
Vacuum-Pro-Black              Hogar
Name: categoria_producto, dtype: object

**Debido a que pudimos comprobar que cada producto pertenece a una sola categoría, es posible imputar los valores con el método de mapeo usado en los pasos anteriores**

In [34]:
orders['categoria_producto'] = orders['categoria_producto'].fillna(orders['nombre_producto'].map(mapeo_categoria))

In [35]:
orders['categoria_producto'].value_counts(dropna=False)

Hogar          8355
Moda           8324
Electronica    8291
NaN              30
Name: categoria_producto, dtype: int64

**Realizando el método de mapeo y rellenando los valores, se obtuvo una disminucion de 80 a 30 valores nulos en la columna de categoría**

In [36]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                24981 non-null  object        
 4   dispositivo         24998 non-null  object        
 5   fuente_referencia   24970 non-null  object        
 6   nombre_producto     24970 non-null  object        
 7   categoria_producto  24970 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         25000 non-null  float64       
 12  monto_esperado      24950 non-null  float64       
 13  diferencia          24950 non-null  float64   

**Después de realizar las imputaciones posibles del dataset orders en las columnas categoricas se redujeron los valores nulos en las siguientes cantidades:**

    -Pais de 300 a 19 datos nulos
    -Dispositivo de 20 a  2 datos nulos
    -Categoria por producto de 80 a 30 datos nulos
    -En las demas columnas categoricas no se determino un proceso seguro para realizar imputaciones sin alterar la informacion


In [37]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [38]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


**Se corrige el formato de fecha ya que se encuentra como object**

**Se imputaron los valores nulos de la columna canal con los valores de id_campaña**

In [39]:
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

In [40]:
marketing['id_campaña'].str.rsplit('_', n=1).str[0]

0           organic
1       paid_search
2            social
3           organic
4       paid_search
           ...     
1615    paid_search
1616         social
1617        organic
1618    paid_search
1619         social
Name: id_campaña, Length: 1620, dtype: object

In [41]:
marketing['canal'] = marketing['canal'].fillna(marketing['id_campaña'].str.rsplit('_', n=1).str[0])

In [42]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


**Se realizó la modificacion del formato de la fecha y se imputaron los valores nulos de la columna canal**

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [43]:

# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)


---

## 🔹 Criterios finales de calidad de datos

Antes de calcular los KPIs financieros se establecieron reglas explícitas para evitar decisiones de limpieza sin respaldo suficiente en los datos.

- **50 registros con `cantidad = NaN`:** se excluyen del análisis financiero. Sin una cantidad válida no es posible calcular de forma consistente el costo asociado mediante `costo_unitario × cantidad`.
- **4 registros con cantidades negativas:** se conservan. Podrían corresponder a devoluciones o ajustes, pero el dataset no contiene una variable que permita confirmarlo; esta interpretación se documenta como una limitación y no como un hecho.
- **10 registros con cantidades extraordinarias de 10,000 y 20,000 unidades:** se conservan. Son valores extremos frente al comportamiento habitual, pero no existe evidencia suficiente para clasificarlos como errores. Podrían corresponder a compras extraordinarias, ventas especiales o promociones dirigidas a clientes específicos.

**Criterio analítico:** un valor atípico no se elimina únicamente por ser extremo. Se conserva cuando no existe evidencia suficiente de error y se documenta su impacto para que los KPIs se interpreten en contexto.


In [ ]:
# Depuración final para el análisis financiero
registros_antes = len(orders)

# Se excluyen únicamente registros sin cantidad.
orders_fin = orders.dropna(subset=["cantidad"]).copy()

print(f"Registros antes de depuración: {registros_antes:,}")
print(f"Registros excluidos por cantidad nula: {registros_antes - len(orders_fin):,}")
print(f"Registros disponibles para análisis financiero: {len(orders_fin):,}")
print(f"Registros con cantidad negativa conservados: {(orders_fin['cantidad'] < 0).sum():,}")
print(f"Registros con cantidad >= 10000 conservados: {(orders_fin['cantidad'] >= 10000).sum():,}")


## 🔹 Paso 2: Análisis de rentabilidad

### 2.1 KPIs financieros

**Objetivo:** evaluar ingresos, costos de producto, inversión en marketing y el resultado económico según los costos disponibles en el modelo.

Para asegurar consistencia, todos los indicadores basados en pedidos se calculan sobre `orders_fin`, que excluye únicamente los 50 registros sin cantidad. Los registros negativos y las compras extraordinarias permanecen en el análisis.


### Cálculo de indicadores


### KPIs financieros con el criterio de depuración definido

Aplicando estas reglas a los datos disponibles:

| KPI | Resultado |
|---|---:|
| Registros analizados | 24,950 |
| Revenue | $51,965,834.26 |
| Costo de producto | $43,124,018.41 |
| Inversión en marketing | $2,871,843.53 |
| Profit según los costos incluidos | $5,969,972.32 |
| Margen sobre revenue | 11.49% |
| Ticket promedio | $2,082.80 |
| Cantidad promedio | 7.12 |

**Interpretación:** los registros extraordinarios de 10,000 y 20,000 unidades permanecen en el análisis, por lo que tienen un impacto considerable sobre revenue, costo y promedios. Los resultados deben interpretarse teniendo presente esta concentración.

> `Profit` representa revenue menos costo de producto e inversión en marketing. No debe interpretarse como utilidad neta contable, ya que el modelo no incorpora todos los posibles gastos operativos, financieros o fiscales.


In [ ]:
# Revenue de los pedidos incluidos en el análisis financiero
revenue = orders_fin["monto_total"].sum()
revenue


In [ ]:
# Incorporar el costo unitario del catálogo
merged = pd.merge(
    orders_fin,
    catalog,
    on="nombre_producto",
    how="inner",
    validate="many_to_one"
)
merged.head()


In [ ]:
# Costo de producto asociado a cada registro
merged["costo_total"] = merged["costo_unitario"] * merged["cantidad"]
merged.head()


In [ ]:
costo_total = merged["costo_total"].sum()
costo_total


In [ ]:
inversion_marketing = marketing["gasto"].sum()
inversion_marketing


In [ ]:
# Resultado según los costos incluidos en el modelo
profit = revenue - costo_total - inversion_marketing
margen = profit / revenue

print(f"Revenue: ${revenue:,.2f}")
print(f"Costo de producto: ${costo_total:,.2f}")
print(f"Inversión en marketing: ${inversion_marketing:,.2f}")
print(f"Profit según costos incluidos: ${profit:,.2f}")
print(f"Margen sobre revenue: {margen:.2%}")


### Interpretación financiera

El modelo obtiene un resultado positivo después de descontar el costo de producto y la inversión en marketing. Sin embargo, este indicador **no debe interpretarse como utilidad neta contable**, ya que el análisis no incorpora todos los posibles gastos operativos, financieros o fiscales.

Los 10 registros de compras extraordinarias se mantienen por decisión metodológica y tienen un impacto relevante sobre los resultados. Por ello, los KPIs deben interpretarse considerando esta concentración y no como evidencia de que todas las operaciones presentan el mismo comportamiento.


### Comportamiento de ventas


In [ ]:
ticket_promedio = orders_fin["monto_total"].mean()
ticket_promedio


In [ ]:
productos_promedio = orders_fin["cantidad"].mean()
productos_promedio


In [ ]:
producto_mas_vendido = (
    orders_fin.groupby("nombre_producto")["cantidad"]
    .sum()
    .idxmax()
)
producto_mas_vendido


In [53]:
gasto_por_canal = marketing.groupby('canal')['gasto'].sum()
gasto_por_canal

canal
organic        972650.96
paid_search    922374.20
social         976818.37
Name: gasto, dtype: float64

---

## 🔹 Paso 3: Funnel de conversión

**Objetivo:** analizar el comportamiento de los usuarios e identificar posibles puntos de abandono.

La tabla `events` registra seis tipos de eventos. El primer análisis cuenta usuarios únicos que realizaron cada evento, independientemente de si completaron previamente las etapas anteriores.

> **Limitación metodológica:** estos conteos no constituyen por sí solos un funnel secuencial. Se observan 7,634 usuarios con `add_to_cart` frente a 7,582 con `select_item`, por lo que algunos usuarios aparecen en una etapa posterior sin estar registrados en la anterior. Esto puede reflejar diferencias de tracking, rutas alternativas de navegación u otra característica no documentada. Con la información disponible no es posible determinar la causa.


### Fuente SQL utilizada durante el desarrollo

Durante el desarrollo original se consultaron tablas alojadas en PostgreSQL. Las credenciales y parámetros de conexión fueron retirados de esta versión pública por seguridad.

Las consultas SQL se conservan como evidencia de la metodología utilizada. Las secciones que dependen de esas tablas no intentan reconectarse automáticamente al servidor original.


```python
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head(10)
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python
query_totals = '''

SELECT COUNT(DISTINCT nombre_evento) AS total_eventos_unicos
FROM events;

'''
totals = pd.read_sql(query_totals, con=engine)
totals
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python
query_totals = '''

SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY case nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;

'''
totals = pd.read_sql(query_totals, con=engine)
totals
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python
# PARTE 1: Totales del funnel
# ======================


query_totals = '''




WITH cte_first_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
),

cte_item AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'

),

cte_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'

),

cte_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
),

cte_payment AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
),

cte_purchase AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'purchase'
)

SELECT
  (SELECT COUNT(*) FROM cte_first_visit)   AS first_visit_user,
  (SELECT COUNT(*) FROM cte_item) AS item_users,
  (SELECT COUNT(*) FROM cte_cart)     AS cart_users,
  (SELECT COUNT(*) FROM cte_checkout) AS checkout_user,
  (SELECT COUNT(*) FROM cte_payment) AS payment_user,
  (SELECT COUNT(*) FROM cte_purchase) AS purchase_user;
 
'''


totals = pd.read_sql(query_totals, con=engine)
totals
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


### Validación de la secuencia

Los conteos de usuarios únicos son:

`first_visit: 7,796 → select_item: 7,582 → add_to_cart: 7,634 → begin_checkout: 7,208 → add_payment_info: 6,250 → purchase: 6,240`

La secuencia no es estrictamente decreciente: `add_to_cart` contiene **52 usuarios más** que `select_item`. Por este motivo, una tasa entre etapas calculada únicamente dividiendo estos totales independientes puede producir una interpretación incorrecta del funnel.

Para un funnel estrictamente secuencial sería necesario reconstruir el recorrido a nivel de usuario y, cuando la información temporal esté disponible, validar también el orden de los eventos.


```python
# Tasas descriptivas entre los conteos agregados de usuarios únicos.
# Se usa 100.0 para evitar división entera.
# IMPORTANTE: estas tasas NO prueban un recorrido secuencial del mismo conjunto de usuarios.

query_conversion = '''
WITH event_counts AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios
    FROM events
    GROUP BY nombre_evento
)
SELECT
    MAX(CASE WHEN nombre_evento = 'first_visit' THEN usuarios END) AS first_visit_user,
    MAX(CASE WHEN nombre_evento = 'select_item' THEN usuarios END) AS item_users,
    MAX(CASE WHEN nombre_evento = 'add_to_cart' THEN usuarios END) AS cart_users,
    MAX(CASE WHEN nombre_evento = 'begin_checkout' THEN usuarios END) AS checkout_user,
    MAX(CASE WHEN nombre_evento = 'add_payment_info' THEN usuarios END) AS payment_user,
    MAX(CASE WHEN nombre_evento = 'purchase' THEN usuarios END) AS purchase_user
FROM event_counts;
'''

conversion = pd.read_sql(query_conversion, con=engine)

stages = [
    ("first_visit_user", "item_users"),
    ("item_users", "cart_users"),
    ("cart_users", "checkout_user"),
    ("checkout_user", "payment_user"),
    ("payment_user", "purchase_user")
]

for current_stage, next_stage in stages:
    current = conversion.loc[0, current_stage]
    nxt = conversion.loc[0, next_stage]
    conversion[f"variacion_{current_stage}_a_{next_stage}_pct"] = (nxt - current) * 100.0 / current

conversion
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


### Interpretación del funnel

A partir de los conteos agregados, la mayor disminución observable ocurre entre `begin_checkout` (7,208 usuarios) y `add_payment_info` (6,250 usuarios): **958 usuarios menos**, equivalente aproximadamente a **13.29%** respecto a quienes iniciaron checkout.

Entre `add_payment_info` y `purchase` la diferencia es de solo 10 usuarios (≈0.16%).

No se interpreta `select_item → add_to_cart` como una conversión superior al 100% ni como un drop-off negativo. La existencia de 52 usuarios adicionales en `add_to_cart` se reporta como una inconsistencia o característica del tracking que requiere investigación.

La relación global entre usuarios con `first_visit` y usuarios con `purchase` es aproximadamente **80.04%**, pero debe describirse como una **relación entre conteos de usuarios únicos**, no como una conversión secuencial confirmada, hasta reconstruir las trayectorias individuales.


## 🔎 Limitaciones y decisiones metodológicas

- **PostgreSQL:** las credenciales fueron eliminadas. Las tablas `events`, `users` y `user_activity` no están disponibles como archivos locales; las consultas SQL se conservan como evidencia metodológica.
- **Funnel:** el análisis original registró 7,796 usuarios en `first_visit`, 7,582 en `select_item`, 7,634 en `add_to_cart`, 7,208 en `begin_checkout`, 6,250 en `add_payment_info` y 6,240 en `purchase`. Como `add_to_cart` supera a `select_item` por 52 usuarios, estos conteos no se presentan como un funnel secuencial validado.
- **Relación first_visit–purchase:** `6,240 / 7,796 ≈ 80.04%` se conserva únicamente como relación descriptiva entre conteos independientes, no como conversión secuencial confirmada.
- **Calidad financiera:** se excluyen 50 registros con `cantidad = NaN`; se conservan 4 cantidades negativas porque podrían representar devoluciones o ajustes, aunque el dataset no lo confirma; y se conservan 10 compras extraordinarias de 10,000 y 20,000 unidades porque no existe evidencia suficiente para clasificarlas como errores.
- **Profit:** representa revenue menos costo de producto e inversión en marketing según las variables disponibles; no equivale a utilidad neta contable.
- **Cohortes:** la consulta original presenta limitaciones en la definición de cohorte, denominador y ventanas semanales; sus resultados se conservan como históricos, no como KPIs definitivos de retención.
- **Prueba A/B:** se utiliza `α = 0.05`; cuando `p > 0.05`, la interpretación correcta es que no existe evidencia suficiente para rechazar H₀.


## 🔹 Paso 4: Retención por cohortes

**Objetivo:** analizar el comportamiento de retención de los usuarios después de su registro.

Durante el desarrollo original se utilizaron las tablas PostgreSQL `users` y `user_activity`. La ejecución histórica produjo cohortes mensuales de enero a junio de 2025.

### Revisión metodológica

Al auditar el SQL original se identificaron limitaciones que impiden presentar los porcentajes obtenidos como una medición definitiva de retención:

1. La cohorte se construyó con `MIN(fecha_actividad)` dentro de `user_activity`, aunque la definición planteada en el proyecto era agrupar por **mes de registro** (`fecha_registro` de `users`).
2. `clientes_iniciales` se calculó con `COUNT(*)` sobre registros de actividad, no con el número único de usuarios pertenecientes a cada cohorte.
3. Las condiciones `dias_despues_registro >= 7`, `>= 14` y `>= 21` son acumulativas y pueden contar actividad posterior dentro de varias métricas, en lugar de medir de forma independiente la actividad de cada semana.

Por estas razones, los resultados históricos se conservan como evidencia del ejercicio SQL, pero **no se utilizan como KPIs definitivos de retención** en el portafolio.

La consulta correcta debería partir de `users.fecha_registro`, unir cada usuario con `user_activity` y calcular usuarios únicos activos en ventanas semanales claramente definidas. Como las tablas originales ya no están disponibles fuera del servidor del bootcamp, esa corrección no puede recalcularse de forma verificable en esta versión pública.


```python
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python
# Retención por cohortes
# ======================

query_cohort_retention_final = '''

SELECT id_usuario,
count(fecha_registro) AS conteo_registros
FROM users
GROUP BY id_usuario
HAVING count(fecha_registro) > 1


'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


**No hay usuarios que se hayan registrado más de una vez**

```python

query_cohort_retention_final = '''

SELECT id_usuario,
       fecha_registro,
       DATE_TRUNC('month', MIN(CAST(fecha_registro AS DATE))) AS cohort_mensual
FROM users
GROUP BY id_usuario, fecha_registro;

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python


query_cohort_retention_final = '''

WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', MIN(CAST(fecha_actividad AS DATE))) AS cohort_mensual,
        dias_despues_registro,
        activo
FROM user_activity
GROUP BY id_usuario, dias_despues_registro, activo
)
SELECT
    cohort_mensual,
    COUNT(*) AS clientes_iniciales,
    COUNT(CASE WHEN dias_despues_registro >= 7 AND activo = 1 THEN 1 END) AS retenido_w1,
    COUNT(CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN 1 END) AS retenido_w2,
    COUNT(CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN 1 END) AS retenido_w3
  FROM cohortes
  GROUP BY cohort_mensual

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


```python
query_cohort_retention_final = '''

WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', MIN(CAST(fecha_actividad AS DATE))) AS cohort_mensual,
        dias_despues_registro,
        activo
FROM user_activity
GROUP BY id_usuario, dias_despues_registro, activo
),
retencion AS (
SELECT
    cohort_mensual,
    COUNT(*) AS clientes_iniciales,
    COUNT(CASE WHEN dias_despues_registro >= 7 AND activo = 1 THEN 1 END) AS retenido_w1,
    COUNT(CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN 1 END) AS retenido_w2,
    COUNT(CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN 1 END) AS retenido_w3
FROM cohortes
GROUP BY cohort_mensual
)
SELECT
  TO_CHAR(cohort_mensual, 'YYYY-MM') AS cohorte, 
  clientes_iniciales,
  ROUND(retenido_w1::numeric / clientes_iniciales, 2) AS semana_1,
  ROUND(retenido_w2::numeric / clientes_iniciales, 2) AS semana_2,
  ROUND(retenido_w3::numeric / clientes_iniciales, 2) AS semana_3
FROM retencion
ORDER BY  cohort_mensual;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final
```

> Consulta conservada como evidencia del análisis original. Requiere las tablas PostgreSQL del entorno del bootcamp para volver a ejecutarse.


### Resultado histórico de la consulta original

La ejecución original produjo los siguientes porcentajes:

| Cohorte | Clientes iniciales* | Semana 1 | Semana 2 | Semana 3 |
|---|---:|---:|---:|---:|
| 2025-01 | 2,694 | 41% | 21% | 9% |
| 2025-02 | 6,050 | 42% | 32% | 21% |
| 2025-03 | 6,387 | 41% | 31% | 21% |
| 2025-04 | 6,340 | 42% | 32% | 21% |
| 2025-05 | 6,715 | 41% | 30% | 20% |
| 2025-06 | 3,814 | 41% | 37% | 29% |

\*`clientes_iniciales` corresponde al cálculo de la consulta original y, tras la auditoría, no se interpreta como tamaño validado de cohorte debido al uso de `COUNT(*)` sobre registros de actividad.

Los valores sugieren diferencias entre cohortes, pero **no se concluye que las cohortes recientes tengan mayor retención**, porque la metodología original requiere corrección antes de realizar esa comparación.


---

### SQL recomendado para una futura reconstrucción de cohortes

Cuando las tablas originales estén disponibles, la cohorte debe definirse desde la fecha de registro y la retención mediante usuarios únicos activos en cada ventana:

```sql
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),
actividad AS (
    SELECT
        c.id_usuario,
        c.cohorte,
        ua.dias_despues_registro,
        ua.activo
    FROM cohortes c
    LEFT JOIN user_activity ua
        ON c.id_usuario = ua.id_usuario
)
SELECT
    TO_CHAR(cohorte, 'YYYY-MM') AS cohorte,
    COUNT(DISTINCT id_usuario) AS usuarios_cohorte,
    COUNT(DISTINCT CASE
        WHEN dias_despues_registro = 7 AND activo = 1 THEN id_usuario
    END)::numeric / NULLIF(COUNT(DISTINCT id_usuario), 0) AS retencion_w1,
    COUNT(DISTINCT CASE
        WHEN dias_despues_registro = 14 AND activo = 1 THEN id_usuario
    END)::numeric / NULLIF(COUNT(DISTINCT id_usuario), 0) AS retencion_w2,
    COUNT(DISTINCT CASE
        WHEN dias_despues_registro = 21 AND activo = 1 THEN id_usuario
    END)::numeric / NULLIF(COUNT(DISTINCT id_usuario), 0) AS retencion_w3
FROM actividad
GROUP BY cohorte
ORDER BY cohorte;
```

> Esta consulta se incluye como propuesta metodológica y no como resultado ejecutado.


## 🔹 Paso 5: Prueba estadística A/B

**Objetivo:** evaluar si la modificación de la interfaz de checkout está asociada con una diferencia estadísticamente significativa en la conversión de compra.

La variable `convirtio` toma el valor 1 cuando el usuario completó la compra y 0 cuando no la completó. Se utiliza una prueba chi-cuadrado de independencia y Cramér's V como medida del tamaño de la asociación.


### Hipótesis estadística

- **H₀:** no existe asociación entre la variante de checkout y la conversión.
- **H₁:** existe asociación entre la variante de checkout y la conversión.

**Prueba:** chi-cuadrado de independencia  
**Nivel de significancia:** α = 0.05 (95% de confianza)


In [66]:

checkout.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [67]:

tabla = pd.crosstab(
    checkout["variante"],
    checkout["convirtio"]
)

tabla


convirtio,0,1
variante,,
control,4186,779
tratamiento,4215,820


In [ ]:
chi2, p, dof, expected = chi2_contingency(tabla)

print(f"Chi-cuadrado: {chi2:.4f}")
print(f"p-value: {p:.4f}")
print(f"Grados de libertad: {dof}")


In [69]:
n = tabla.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(tabla.shape) -1)))
cramers_v 

0.007859934897509835

### Resultado validado de la prueba A/B

La tabla de contingencia guardada en el notebook contiene:

| Variante | No convirtió | Convirtió | Total | Tasa de conversión |
|---|---:|---:|---:|---:|
| Control | 4,186 | 779 | 4,965 | **15.69%** |
| Tratamiento | 4,215 | 820 | 5,035 | **16.29%** |

La variante de tratamiento presenta una diferencia observada de aproximadamente **+0.60 puntos porcentuales** frente al control (≈3.8% de incremento relativo). Sin embargo, la prueba chi-cuadrado obtiene:

- **χ² = 0.6178**
- **p = 0.4319**
- **gl = 1**
- **Cramér's V = 0.0079**

Con `α = 0.05`, `p > 0.05`; por lo tanto, **no se rechaza H₀**. Con estos datos no existe evidencia estadísticamente significativa de asociación entre la variante de checkout y la conversión.

Cramér's V es cercano a cero, lo que indica que la asociación observada entre variante y conversión es de magnitud muy pequeña.

**Conclusión de negocio:** aunque el tratamiento muestra una conversión descriptivamente mayor (16.29% vs. 15.69%), la evidencia disponible no permite atribuir esa diferencia al cambio de interfaz. No se recomienda interpretar el incremento observado como un efecto demostrado del tratamiento.


---

## 🔹 Paso 6: Comunicación de resultados en Power BI

**Objetivo:** presentar de forma visual los principales resultados de ventas, costos, marketing, conversión y rentabilidad.

Los datasets limpios se utilizan como fuente para construir el dashboard y facilitar la interpretación de los KPIs y hallazgos del proyecto.


---

## 🚀 Entrega final

El proyecto se complementa con un dashboard desarrollado en **Power BI**, disponible en el repositorio junto con este notebook.


### Dashboard

El archivo `.pbix` del proyecto se encuentra disponible en el repositorio para consultar el dashboard completo.
